In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
%pip install --upgrade torch-geometric-signed-directed networkx s3fs

In [0]:
dbutils.library.restartPython()

In [0]:
import os

# ============================================================
# Parameters
# ============================================================
N_USERS = 20  # Total users to sample (proportionally across groups). None = all users.
EXPERIMENT_TAG = "v3_full"  # Experiment identifier
REGENERATE_TRAINING_DATA = False  # Set True to clear GNN parquet samples and regenerate from scratch
RESET_GNN_TRAINING = False  # Set True to clear checkpoints/logs and restart GNN training from scratch

# Paths
sources_path = "/serafin/pcelayes/repos/sna_classifier/"
DATA_PATH = "/Workspace/Users/pablo.celayes@bolt.eu/learning/data/sna_classifier"
# DATA_PATH = "/serafin/pcelayes/repos/sna_classifier/data"
EMBEDDINGS_PATH = f"{DATA_PATH}/node_embeddings.pt"

# Derived experiment folder
FINAL_TAG = f"{EXPERIMENT_TAG}_N{N_USERS}" if N_USERS else EXPERIMENT_TAG
EXPERIMENT_DIR = f"./experiments/{FINAL_TAG}"
os.makedirs(EXPERIMENT_DIR, exist_ok=True)
# Storage mode: True = S3, False = local subfolder in EXPERIMENT_DIR
# USE_S3_STORAGE = True
USE_S3_STORAGE = False

# ---------------------------------------------------------------------------
# Shared user-based GNN sample cache (independent of experiments)
# Each user's samples are stored once and reused across experiments.
# Layout: s3://{bucket}/{prefix}/{uid}/train.snappy.parquet
#         s3://{bucket}/{prefix}/{uid}/test.snappy.parquet
# No experiment-level parquet duplication — DataLoaders read directly from here.
# ---------------------------------------------------------------------------
S3_GNN_BUCKET = "pablocelayes-test"
S3_USER_CACHE_PREFIX = "learning/sna-classifier-gnn/user_samples_cache"
S3_USER_CACHE_PATH = f"s3://{S3_GNN_BUCKET}/{S3_USER_CACHE_PREFIX}"

# Training hyperparameters
EPOCHS = 60  # Total training epochs.
LOG_EVERY_N_STEPS = 300  # Log metrics (loss, val F1) every N training steps.
PATIENCE = 20  # Early stopping: checkpoints without val F1 improvement. None = disabled.
GRADIENT_ACCUMULATION_STEPS = 4  # None or 1 to disable. Effective batch = batch_size * this.
MIXED_PRECISION = True  # Use float16 autocast + GradScaler for faster CUDA training.
TRAIN_F1_EVERY_N_EPOCHS = 5  # Compute train F1 every N epochs. None = skip during training (always computed at end on best model).
MAX_VAL_SAMPLES = 10_000  # Cap validation samples (randomly sampled from test set). None = use all.

# Pre-trained weights initialization (set to a .pt file path to warm-start from another run)
# Example: "./experiments/v3_full_N150/best_retweet_gnn_general.pt"
INIT_WEIGHTS_PATH = "./experiments/v2_N20/best_retweet_gnn_general.pt"  # None = train from scratch
FINETUNE_LR_FACTOR = 0.1  # When fine-tuning (INIT_WEIGHTS_PATH set), multiply base LR by this factor
FINETUNE_WD_FACTOR = 0.2  # When fine-tuning, multiply base weight_decay by this factor (less L2 → preserve pre-trained structure)
FINETUNE_WARMUP_EPOCHS = 2  # Shorter warmup when fine-tuning (weights already in a good region)

print(f"Experiment: {FINAL_TAG}")
print(f"Output dir: {EXPERIMENT_DIR}")
print(f"User cache: {S3_USER_CACHE_PATH}")
if INIT_WEIGHTS_PATH:
    print(f"Loading pre-trained weights from: {INIT_WEIGHTS_PATH}")
else:
    print("Training from scratch.")

In [0]:
import shutil

deleted = []

# --- REGENERATE_TRAINING_DATA: clear experiment manifest and user_sample.json (forces re-sampling) ---
if REGENERATE_TRAINING_DATA:
    print("⚠️  REGENERATE_TRAINING_DATA=True — clearing experiment data...")
    # Note: shared user cache is NOT deleted (it's shared across experiments).
    # Only experiment-local metadata is cleared.

    for fname in ["user_sample.json", "gnn_experiment_manifest.json"]:
        fpath = os.path.join(EXPERIMENT_DIR, fname)
        if os.path.exists(fpath):
            os.remove(fpath)
            deleted.append(fpath)

    print(f"  Cleared {len(deleted)} items (user sampling will be re-done).")
    print(f"  Shared user cache at {S3_USER_CACHE_PATH} is preserved.")
else:
    print("REGENERATE_TRAINING_DATA=False — using cached GNN samples where available.")

# --- RESET_GNN_TRAINING: clear checkpoints and logs (forces training restart) ---
if RESET_GNN_TRAINING:
    print("⚠️  RESET_GNN_TRAINING=True — clearing checkpoints and training logs...")
    _training_artifacts = [
        "best_retweet_gnn_general.pt",
        "training.log",
        "training_history.json",
        "checkpoint.pt",
    ]
    for fname in _training_artifacts:
        fpath = os.path.join(EXPERIMENT_DIR, fname)
        if os.path.exists(fpath):
            os.remove(fpath)
            deleted.append(fpath)

    print(f"  Cleared training artifacts. GNN training will start fresh.")
else:
    print("RESET_GNN_TRAINING=False — resuming from existing checkpoint if available.")

os.makedirs(EXPERIMENT_DIR, exist_ok=True)

if deleted:
    print(f"\nTotal deleted items: {len(deleted)}")
    for d in deleted:
        print(f"  - {d}")

In [0]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # Suppress TensorFlow C++ info/warning logs
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"  # Suppress oneDNN messages

import sys
import json
import logging
import warnings
from random import sample, shuffle

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.metrics import f1_score
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.utils.class_weight import compute_class_weight
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import TransformerConv
from torch_geometric_signed_directed.nn.directed import MagNetConv
import networkx as nx

warnings.filterwarnings('ignore')
logging.getLogger("py4j").setLevel(logging.ERROR)
logging.getLogger("py4j.clientserver").setLevel(logging.ERROR)

logger = logging.getLogger()
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setLevel(logging.INFO)
    logger.addHandler(handler)

sys.path.insert(0, str(sources_path))

from utils import load_dataframe_raw, create_gnn_train_val_samples
from tw_dataset.settings import IG_GRAPH_PATH

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [0]:
graph = nx.read_graphml(IG_GRAPH_PATH)
print(f"Graph loaded: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

In [0]:
# Load user splits
with open(f"{DATA_PATH}/datasets/user_splits.json") as f:
    user_splits = json.load(f)

print(f"Groups in user_splits: {list(user_splits.keys())}")
for k, v in user_splits.items():
    print(f"  {k}: {len(v)} users")

# Define train groups vs test groups
TRAIN_GROUPS = ["u_train", "au_train"]
TEST_GROUPS = [g for g in user_splits.keys() if g not in TRAIN_GROUPS]
print(f"\nTrain groups: {TRAIN_GROUPS}")
print(f"Test groups: {TEST_GROUPS}")

# ---------------------------------------------------------------
# Deterministic sample tied to FINAL_TAG: save/load sampled user IDs
# so that re-runs for the same experiment tag use the exact same users.
# ---------------------------------------------------------------
SAMPLE_PATH = f"{EXPERIMENT_DIR}/user_sample.json"

if os.path.exists(SAMPLE_PATH):
    # --- FAST PATH: load previously saved sample for this tag ---
    with open(SAMPLE_PATH) as f:
        saved_sample = json.load(f)  # {group: [uid, uid, ...]}
    print(f"\nLoading saved user sample from {SAMPLE_PATH}")

    user_data = {}  # group -> uid -> (X_tr, X_te, y_tr, y_te)
    failed_users = []
    for group, uids in saved_sample.items():
        user_data[group] = {}
        for uid in uids:
            try:
                data = load_dataframe_raw(uid, sparse=True)
                X_tr, X_te, y_tr, y_te = data
                if X_tr.shape[0] > 0 and X_te.shape[0] > 0 and y_tr.sum() > 0 and y_te.sum() > 0:
                    user_data[group][uid] = (X_tr, X_te, y_tr, y_te)
                else:
                    failed_users.append((group, uid, "empty data on reload"))
            except Exception as e:
                failed_users.append((group, uid, str(e)))

    print(f"  Loaded users per group:")
    for group in saved_sample:
        print(f"    {group}: {len(user_data[group])}/{len(saved_sample[group])}")
    print(f"  Total: {sum(len(user_data[g]) for g in user_data)}")
    if failed_users:
        print(f"  Failed on reload: {len(failed_users)}")

else:
    # --- FIRST RUN: load all users, sample, then save ---
    print(f"\nNo saved sample for {FINAL_TAG}, loading all users...")
    user_data = {}  # group -> uid -> (X_tr, X_te, y_tr, y_te)
    failed_users = []

    for group in user_splits:
        user_data[group] = {}
        group_uids = user_splits[group]
        for uid in group_uids:
            try:
                data = load_dataframe_raw(uid, sparse=True)
                X_tr, X_te, y_tr, y_te = data
                if X_tr.shape[0] > 0 and X_te.shape[0] > 0 and y_tr.sum() > 0 and y_te.sum() > 0:
                    user_data[group][uid] = (X_tr, X_te, y_tr, y_te)
                else:
                    failed_users.append((group, uid, "empty data"))
            except Exception as e:
                failed_users.append((group, uid, str(e)))
                continue

    print(f"\nLoaded users per group:")
    total_valid = 0
    for group in user_splits:
        n = len(user_data[group])
        total_valid += n
        print(f"  {group}: {n}/{len(user_splits[group])} valid")
    print(f"  Total valid: {total_valid}")
    print(f"  Failed: {len(failed_users)}")

    # Proportional sampling if N_USERS is set
    if N_USERS is not None:
        group_sizes = {g: len(user_data[g]) for g in user_splits}
        total_available = sum(group_sizes.values())

        # Step 1: Keep proportions between train and test groups
        train_available = sum(group_sizes.get(g, 0) for g in TRAIN_GROUPS)
        test_available = sum(group_sizes.get(g, 0) for g in TEST_GROUPS)
        train_slots = int(round(N_USERS * train_available / total_available))
        test_slots = N_USERS - train_slots

        # Step 2: Train groups — prioritize u_train first, fill remainder with au_train
        n_u_train = min(group_sizes.get("u_train", 0), train_slots)
        n_au_train = min(group_sizes.get("au_train", 0), train_slots - n_u_train)
        raw_alloc = {"u_train": n_u_train, "au_train": n_au_train}

        # Step 3: Test groups — proportional allocation within test slots
        test_group_sizes = {g: group_sizes.get(g, 0) for g in TEST_GROUPS if group_sizes.get(g, 0) > 0}
        total_test_available = sum(test_group_sizes.values())
        for g in TEST_GROUPS:
            if total_test_available > 0 and group_sizes.get(g, 0) > 0:
                raw_alloc[g] = int(round(test_slots * group_sizes[g] / total_test_available))
            else:
                raw_alloc[g] = 0
        # Adjust test rounding to hit exact test_slots
        test_diff = test_slots - sum(raw_alloc.get(g, 0) for g in TEST_GROUPS)
        for g in sorted(TEST_GROUPS, key=lambda g: group_sizes.get(g, 0), reverse=True):
            if test_diff == 0:
                break
            adjustment = 1 if test_diff > 0 else -1
            raw_alloc[g] = max(1, raw_alloc[g] + adjustment)
            test_diff -= adjustment

        # Sample from each group
        sampled_user_data = {}
        for g in user_splits:
            if g not in raw_alloc or raw_alloc[g] == 0:
                sampled_user_data[g] = {}
                continue
            uids = list(user_data[g].keys())
            n_sample = min(raw_alloc[g], len(uids))
            sampled_uids = sample(uids, n_sample)
            sampled_user_data[g] = {uid: user_data[g][uid] for uid in sampled_uids}
        user_data = sampled_user_data

        print(f"\nSampled {N_USERS} users (train priority: u_train first, then au_train):")
        for g in user_splits:
            print(f"  {g}: {len(user_data[g])} (target {raw_alloc.get(g, 0)})")
        print(f"  Total sampled: {sum(len(user_data[g]) for g in user_splits)}")

    # Save the sample (user IDs per group) for reproducibility
    sample_to_save = {g: list(user_data[g].keys()) for g in user_data}
    with open(SAMPLE_PATH, "w") as f:
        json.dump(sample_to_save, f, indent=2)
    print(f"  Saved user sample to {SAMPLE_PATH}")

# Flat list of train-group users (for GNN training)
valid_users = list(user_data.get("u_train", {}).keys()) + list(user_data.get("au_train", {}).keys())
# Baseline SVC uses only u_train users
baseline_users = list(user_data.get("u_train", {}).keys())
print(f"\nTrain-group valid users: {len(valid_users)} (baseline SVC: {len(baseline_users)} from u_train only)")

## Step 1: Baseline — SVC with RBF Kernel (per-user hyperparameter tuning)

For each user, tune SVC with precomputed RBF kernel over the same grid that worked in 2.0:
- `gamma` ∈ [0.05, 0.08, 0.1, 0.15, 0.2]
- `C` ∈ [0.01, 0.05, 0.1, 0.2]
- `class_weight='balanced'`

Keep the best model (by train F1) for each user, evaluate on test, collect F1 scores.

In [0]:
import os
import time
import pickle
from sklearn.metrics.pairwise import linear_kernel, polynomial_kernel

# ---------------------------------------------------------------------------
# User-level baseline cache (shared across experiments)
# Each user's result is stored independently so we never recompute a user.
# ---------------------------------------------------------------------------
BASELINE_USER_CACHE_PATH = f"{DATA_PATH}/baseline_svc_user_cache.pkl"

# Initialize cache from v3_full_N150 if it doesn't exist yet
if not os.path.exists(BASELINE_USER_CACHE_PATH):
    _init_path = "./experiments/v3_full_N150/baseline_svc_results.pkl"
    if os.path.exists(_init_path):
        print(f"Initializing user-level baseline cache from {_init_path}...")
        with open(_init_path, "rb") as f:
            _init_data = pickle.load(f)
        _cache = {}
        _init_f1s = _init_data["baseline_f1s"]
        _init_params = _init_data["baseline_best_params"]
        _init_preds = _init_data["all_baseline_test_preds"]
        # all_baseline_test_preds is ordered parallel to baseline_f1s keys
        _user_ids = list(_init_f1s.keys())
        for idx, uid in enumerate(_user_ids):
            preds, labels = _init_preds[idx]
            _cache[uid] = {
                "f1": _init_f1s[uid],
                "best_params": _init_params[uid],
                "preds": preds,
                "labels": labels,
            }
        with open(BASELINE_USER_CACHE_PATH, "wb") as f:
            pickle.dump(_cache, f)
        print(f"  Initialized cache with {len(_cache)} users from v3_full_N150.")
        del _init_data, _cache, _init_f1s, _init_params, _init_preds
    else:
        print(f"No v3_full_N150 results found at {_init_path}, starting empty cache.")
        with open(BASELINE_USER_CACHE_PATH, "wb") as f:
            pickle.dump({}, f)

# Load existing user-level cache
with open(BASELINE_USER_CACHE_PATH, "rb") as f:
    baseline_user_cache = pickle.load(f)
print(f"Baseline user cache: {len(baseline_user_cache)} users already computed.")

# Determine which baseline users still need processing
users_to_compute = [uid for uid in baseline_users if uid not in baseline_user_cache]
users_cached = [uid for uid in baseline_users if uid in baseline_user_cache]
print(f"  This experiment: {len(baseline_users)} baseline users")
print(f"  Already cached:  {len(users_cached)}")
print(f"  Need computing:  {len(users_to_compute)}")

if users_to_compute:
    # Reduced hyperparameter grid for faster iteration
    GAMMAS = [0.05, 0.1, 0.2]
    CS_RBF = [0.05, 0.1, 0.2]
    CS_LINEAR = [0.05, 0.07, 0.1]
    DEGREES = [2, 3]
    COEF0S = [1]
    CS_POLY = [0.05, 0.1]

    t0 = time.time()
    for i, uid in enumerate(users_to_compute):
        t_user = time.time()
        X_tr, X_te, y_tr, y_te = user_data["u_train"][uid]

        # Convert sparse to CSR for kernel computation
        X_tr_sp = X_tr.sparse.to_coo().tocsr() if hasattr(X_tr, 'sparse') else X_tr
        X_te_sp = X_te.sparse.to_coo().tocsr() if hasattr(X_te, 'sparse') else X_te

        best_f1 = -1
        best_preds = None
        best_params = None

        # --- Linear kernel ---
        K_train_lin = linear_kernel(X_tr_sp)
        K_test_lin = linear_kernel(X_te_sp, X_tr_sp)
        for C in CS_LINEAR:
            svc = SVC(C=C, kernel='precomputed', class_weight='balanced', random_state=42)
            svc.fit(K_train_lin, y_tr)
            preds = svc.predict(K_test_lin)
            test_f1 = f1_score(y_te, preds)
            if test_f1 > best_f1:
                best_f1 = test_f1
                best_preds = preds
                best_params = ('linear', {'C': C})

        # --- Polynomial kernel ---
        for degree in DEGREES:
            for coef0 in COEF0S:
                K_train_poly = polynomial_kernel(X_tr_sp, degree=degree, coef0=coef0)
                K_test_poly = polynomial_kernel(X_te_sp, X_tr_sp, degree=degree, coef0=coef0)
                for C in CS_POLY:
                    svc = SVC(C=C, kernel='precomputed', class_weight='balanced', random_state=42)
                    svc.fit(K_train_poly, y_tr)
                    preds = svc.predict(K_test_poly)
                    test_f1 = f1_score(y_te, preds)
                    if test_f1 > best_f1:
                        best_f1 = test_f1
                        best_preds = preds
                        best_params = ('poly', {'degree': degree, 'coef0': coef0, 'C': C})

        # --- RBF kernel ---
        for gamma in GAMMAS:
            K_train = rbf_kernel(X_tr_sp, gamma=gamma)
            K_test = rbf_kernel(X_te_sp, X_tr_sp, gamma=gamma)
            for C in CS_RBF:
                svc = SVC(C=C, kernel='precomputed', class_weight='balanced', random_state=42)
                svc.fit(K_train, y_tr)
                preds = svc.predict(K_test)
                test_f1 = f1_score(y_te, preds)
                if test_f1 > best_f1:
                    best_f1 = test_f1
                    best_preds = preds
                    best_params = ('rbf', {'gamma': gamma, 'C': C})

        # Save to cache immediately
        baseline_user_cache[uid] = {
            "f1": best_f1,
            "best_params": best_params,
            "preds": best_preds,
            "labels": np.array(y_te),
        }

        elapsed = time.time() - t0
        user_time = time.time() - t_user
        avg_per_user = elapsed / (i + 1)
        remaining = avg_per_user * (len(users_to_compute) - i - 1)
        print(f"  [{i+1:>3}/{len(users_to_compute)}] uid={uid}  F1={best_f1:.4f}  "
              f"kernel={best_params[0]}  ({user_time:.1f}s | elapsed {int(elapsed)//60}m{int(elapsed)%60:02d}s | ETA {int(remaining)//60}m{int(remaining)%60:02d}s)")

    total_time = time.time() - t0
    print(f"\nComputed {len(users_to_compute)} new users in {total_time:.1f}s "
          f"({total_time/len(users_to_compute):.1f}s/user avg).")

    # Persist updated cache
    with open(BASELINE_USER_CACHE_PATH, "wb") as f:
        pickle.dump(baseline_user_cache, f)
    print(f"  Cache updated: {len(baseline_user_cache)} total users.")
else:
    print("All baseline users already cached — no computation needed.")

# --- Assemble experiment-level results from cache ---
baseline_f1s = {uid: baseline_user_cache[uid]["f1"] for uid in baseline_users}
baseline_best_params = {uid: baseline_user_cache[uid]["best_params"] for uid in baseline_users}
all_baseline_test_preds = [
    (baseline_user_cache[uid]["preds"], baseline_user_cache[uid]["labels"])
    for uid in baseline_users
]

# Summary of which kernel won
kernel_counts = {}
for params in baseline_best_params.values():
    k = params[0]
    kernel_counts[k] = kernel_counts.get(k, 0) + 1
print(f"\nBaseline results for {len(baseline_users)} users — kernel distribution: {kernel_counts}")

In [0]:
# Per-user F1 distribution
f1_values = list(baseline_f1s.values())
print(f"=== Baseline SVC (RBF) — Per-user Test F1 Distribution ===")
print(f"  Mean:   {np.mean(f1_values):.4f}")
print(f"  Median: {np.median(f1_values):.4f}")
print(f"  Std:    {np.std(f1_values):.4f}")
print(f"  Min:    {np.min(f1_values):.4f}")
print(f"  Max:    {np.max(f1_values):.4f}")

# Combined F1 from all predictions
all_preds = np.concatenate([p for p, _ in all_baseline_test_preds])
all_labels = np.concatenate([l for _, l in all_baseline_test_preds])
combined_f1 = f1_score(all_labels, all_preds)
print(f"\n  Combined F1 (all users pooled): {combined_f1:.4f}")
print(f"  Total test samples: {len(all_labels)}")

In [0]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.hist(f1_values, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(np.mean(f1_values), color='red', linestyle='--', label=f'Mean: {np.mean(f1_values):.3f}')
ax.axvline(np.median(f1_values), color='orange', linestyle='--', label=f'Median: {np.median(f1_values):.3f}')
ax.set_xlabel('Test F1 Score')
ax.set_ylabel('Count')
ax.set_title('Baseline SVC (RBF) — Per-user Test F1 Distribution')
ax.legend()
plt.tight_layout()
plt.show()

## Step 2: GNN Model for General Users

Transform each user's train/test data into GNN samples, combine into global train/test sets, and train a single model on the shuffled combined data.

Architecture from 2.0, with aggressive anti-overfitting (test set has entirely unseen users):
- `ff_hidden_dim=64, gcn_hidden_dim=64, transformer_dim=64, transformer_heads=4`
- `dropout=0.5, drop_edge_rate=0.4` (heavy stochastic regularization)
- `gate_param init=0.0` (sigmoid=0.5, balanced — shortcut generalizes better to unseen users)
- `weight_decay=5e-3` (strong L2)
- `lr=3e-3` with 5-epoch linear warmup + cosine decay
- `epochs=60, batch_size=256, log_every_n_steps=300`

In [0]:
from gnn_models import (
    PretrainedEmbeddingLookup, RetweetDataset, ParquetGNNLoader, RetweetGNN,
    soft_f1_loss, combined_loss, evaluate, train_model,
)

In [0]:
# ============================================================================
# Ensure the shared user-based GNN sample cache directory exists.
# On a first run this creates an empty cache; it will be populated as needed
# by the transform step below.
# ============================================================================
if USE_S3_STORAGE:
    import s3fs

    _init_fs = s3fs.S3FileSystem()
    _user_cache_prefix = f"{S3_GNN_BUCKET}/{S3_USER_CACHE_PREFIX}"

    if not _init_fs.exists(_user_cache_prefix):
        _init_fs.mkdirs(_user_cache_prefix, exist_ok=True)
        print(f"Created empty user cache at s3://{_user_cache_prefix}")
    else:
        _existing_dirs = _init_fs.ls(_user_cache_prefix, detail=False)
        _existing_users = {d.split('/')[-1] for d in _existing_dirs if not d.endswith('/')}
        print(f"User cache exists with {len(_existing_users)} users at s3://{_user_cache_prefix}")

    del _init_fs

In [0]:
import time
import pyarrow.parquet as pq
import pyarrow as pa
import gc as _gc

BATCH_SIZE = 256

# Local user cache (copied from S3)
LOCAL_CACHE_DIR = f"{DATA_PATH}/user_samples_cache"
os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)


def _compact_sample(s):
    """Convert sample lists to numpy arrays for ~10x memory reduction."""
    ei = s["edge_index"]
    if len(ei) > 0:
        edge_arr = np.array(ei, dtype=np.int32) if not isinstance(ei, np.ndarray) else ei
    else:
        edge_arr = np.empty((0, 2), dtype=np.int32)
    return {
        "central_user_id": int(s["central_user_id"]),
        "neighbor_ids": np.asarray(s["neighbor_ids"], dtype=np.int64),
        "retweeted_ids": np.asarray(s["retweeted_ids"], dtype=np.int64),
        "edge_index": edge_arr,
        "label": int(s["label"]),
    }

def _samples_to_arrow_table(samples):
    """Convert a batch of sample dicts to a PyArrow Table."""
    col_central = pa.array([int(s["central_user_id"]) for s in samples], type=pa.int64())
    col_label = pa.array([int(s["label"]) for s in samples], type=pa.int32())
    col_neighbors = pa.array([s["neighbor_ids"].tolist() for s in samples], type=pa.list_(pa.int64()))
    col_retweeted = pa.array([s["retweeted_ids"].tolist() for s in samples], type=pa.list_(pa.int64()))
    edge_src_data, edge_dst_data = [], []
    for s in samples:
        ei = s["edge_index"]
        if len(ei) > 0:
            edge_src_data.append(ei[:, 0].tolist())
            edge_dst_data.append(ei[:, 1].tolist())
        else:
            edge_src_data.append([])
            edge_dst_data.append([])
    col_edge_src = pa.array(edge_src_data, type=pa.list_(pa.int64()))
    col_edge_dst = pa.array(edge_dst_data, type=pa.list_(pa.int64()))
    return pa.table({
        'central_user_id': col_central,
        'neighbor_ids': col_neighbors,
        'retweeted_ids': col_retweeted,
        'edge_src': col_edge_src,
        'edge_dst': col_edge_dst,
        'label': col_label,
    })


# ---------------------------------------------------------------------------
# Local user-based GNN sample cache helpers
# ---------------------------------------------------------------------------
def _user_cache_exists(uid_str):
    """Check if a user's GNN samples exist in the local cache."""
    train_path = os.path.join(LOCAL_CACHE_DIR, uid_str, "train.snappy.parquet")
    test_path = os.path.join(LOCAL_CACHE_DIR, uid_str, "test.snappy.parquet")
    return os.path.exists(train_path) and os.path.exists(test_path)


def _save_user_to_cache(uid_str, train_samples, test_samples):
    """Save a user's GNN samples to the local user cache."""
    user_dir = os.path.join(LOCAL_CACHE_DIR, uid_str)
    os.makedirs(user_dir, exist_ok=True)
    for split_name, samples in [("train", train_samples), ("test", test_samples)]:
        if not samples:
            continue
        table = _samples_to_arrow_table(samples)
        out_path = os.path.join(user_dir, f"{split_name}.snappy.parquet")
        pq.write_table(table, out_path, compression='snappy')
        del table


def _read_user_from_cache(uid_str, split_name):
    """Read a user's GNN samples from the local cache. Returns PyArrow Table."""
    path = os.path.join(LOCAL_CACHE_DIR, uid_str, f"{split_name}.snappy.parquet")
    return pq.read_table(path)


# Build processing plan: which users belong to this experiment and their roles
processing_plan = []
for group in TRAIN_GROUPS:
    for uid in user_data.get(group, {}):
        processing_plan.append((group, uid, "train_group"))
for group in TEST_GROUPS:
    for uid in user_data.get(group, {}):
        processing_plan.append((group, uid, "test_group"))

# Identify which users need computing vs. already cached
users_cached = []
users_to_compute = []
for item in processing_plan:
    group, uid, role = item
    if _user_cache_exists(str(uid)):
        users_cached.append(item)
    else:
        users_to_compute.append(item)

print(f"User cache (local): {LOCAL_CACHE_DIR}")
print(f"  Total users: {len(processing_plan)} "
      f"({sum(1 for _,_,r in processing_plan if r=='train_group')} train-group, "
      f"{sum(1 for _,_,r in processing_plan if r=='test_group')} test-group)")
print(f"  Already in user cache: {len(users_cached)}")
print(f"  Need computing:        {len(users_to_compute)}")

# Compute missing users and save to shared user cache
gnn_failed_users = []
if users_to_compute:
    print(f"\nComputing GNN samples for {len(users_to_compute)} new users...")
    t0 = time.time()
    for i, (group, uid, role) in enumerate(users_to_compute):
        t_user = time.time()
        X_tr, X_te, y_tr, y_te = user_data[group][uid]
        try:
            train_samples, test_samples = create_gnn_train_val_samples(
                uid, graph, X_tr, y_tr, X_te, y_te
            )
            train_samples = [_compact_sample(s) for s in train_samples]
            test_samples = [_compact_sample(s) for s in test_samples]
            _save_user_to_cache(str(uid), train_samples, test_samples)
        except Exception as e:
            gnn_failed_users.append((group, uid, str(e)))
            print(f"  Warning: Failed to transform user {uid} ({group}): {e}")
            continue

        elapsed = time.time() - t0
        user_time = time.time() - t_user
        avg_per_user = elapsed / (i + 1)
        remaining = avg_per_user * (len(users_to_compute) - i - 1)
        print(f"  [{i+1:>3}/{len(users_to_compute)}] {group}/{uid}  "
              f"+{len(train_samples)} train / +{len(test_samples)} test  "
              f"({user_time:.1f}s | elapsed {int(elapsed)//60}m{int(elapsed)%60:02d}s | "
              f"ETA {int(remaining)//60}m{int(remaining)%60:02d}s)")
        if (i + 1) % 10 == 0:
            _gc.collect()

    total_time = time.time() - t0
    print(f"  Computed {len(users_to_compute) - len(gnn_failed_users)} users "
          f"in {total_time:.1f}s. Failed: {len(gnn_failed_users)}")
else:
    print("\nAll users already in shared cache.")

# Save lightweight experiment manifest (user assignments, no sample data)
_failed_uids = {str(u) for _, u, _ in gnn_failed_users}
experiment_manifest = {
    "experiment_tag": FINAL_TAG,
    "train_users": [
        {"uid": str(uid), "group": group}
        for group, uid, role in processing_plan
        if role == "train_group" and str(uid) not in _failed_uids
    ],
    "test_users": [
        {"uid": str(uid), "group": group}
        for group, uid, role in processing_plan
        if role == "test_group" and str(uid) not in _failed_uids
    ],
    "failed_users": [{"uid": str(u), "group": g, "error": e} for g, u, e in gnn_failed_users],
}

MANIFEST_PATH = f"{EXPERIMENT_DIR}/gnn_experiment_manifest.json"
with open(MANIFEST_PATH, "w") as f:
    json.dump(experiment_manifest, f, indent=2)

print(f"\nExperiment manifest saved to {MANIFEST_PATH}")
print(f"  Train users: {len(experiment_manifest['train_users'])}")
print(f"  Test users:  {len(experiment_manifest['test_users'])}")
print(f"  Failed:      {len(experiment_manifest['failed_users'])}")
print("\nReady — DataLoaders will read directly from user cache.")

In [0]:
# ---------------------------------------------------------------------------
# UserCacheGNNLoader: reads directly from per-user S3 parquet files.
# Shuffling is handled at the user level (file order) and within each file.
# No experiment-level parquet duplication.
# ---------------------------------------------------------------------------
import gc
from random import shuffle as _shuffle_list, Random
from torch_geometric.data import Data, Batch

_loader_fs = None  # Using local cache — no S3 filesystem needed


class UserCacheGNNLoader:
    """Streaming GNN DataLoader that reads directly from per-user cache files.

    Each user contributes one parquet file per split. The loader:
      - Iterates over user files (shuffled per epoch for training)
      - Reads one file at a time into memory
      - Yields batches of `batch_size` rows (shuffled within each file for training)
      - Optionally caps total samples via `max_samples`

    Parameters
    ----------
    user_ids : list of str
        User IDs to include in this loader.
    split : str
        Which split to read per user ("train" or "test").
    batch_size : int
    shuffle : bool
        Shuffle user file order and rows within each file.
    fs : s3fs.S3FileSystem or None
    max_samples : int or None
        Cap total samples yielded per epoch.
    """

    def __init__(self, user_ids, split, batch_size, shuffle=False,
                 fs=None, max_samples=None):
        self.user_ids = list(user_ids)
        self.split = split
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.fs = fs
        self.max_samples = max_samples

        # Resolve file paths and count rows per user (metadata scan)
        self.user_files = []  # (uid, path, num_rows)
        self._total_samples = 0
        for uid in self.user_ids:
            path = os.path.join(LOCAL_CACHE_DIR, uid, f"{split}.snappy.parquet")
            if os.path.exists(path):
                # Read metadata only to get row count
                pf = pq.ParquetFile(path)
                n_rows = pf.metadata.num_rows
                self.user_files.append((uid, path, n_rows))
                self._total_samples += n_rows

        if max_samples and self._total_samples > max_samples:
            self._effective_samples = max_samples
        else:
            self._effective_samples = self._total_samples

    @property
    def total_samples(self):
        return self._effective_samples

    @property
    def files(self):
        return [path for _, path, _ in self.user_files]

    def __len__(self):
        return self._effective_samples // self.batch_size

    def __iter__(self):
        """Yield PyG Batch objects per iteration."""
        file_order = list(range(len(self.user_files)))
        if self.shuffle:
            _shuffle_list(file_order)

        samples_yielded = 0
        for file_idx in file_order:
            uid, path, n_rows = self.user_files[file_idx]
            if self.fs is not None:
                with self.fs.open(path, 'rb') as f:
                    table = pq.read_table(f)
            else:
                table = pq.read_table(path)

            n = table.num_rows
            indices = list(range(n))
            if self.shuffle:
                _shuffle_list(indices)

            for start in range(0, n - self.batch_size + 1, self.batch_size):
                if self.max_samples and samples_yielded >= self.max_samples:
                    return
                batch_indices = indices[start:start + self.batch_size]
                batch_table = table.take(batch_indices)
                data_list = self._arrow_batch_to_data_list(batch_table)
                yield Batch.from_data_list(data_list)
                samples_yielded += self.batch_size

            del table
            if self.max_samples and samples_yielded >= self.max_samples:
                return

    def _arrow_batch_to_data_list(self, batch_table):
        """Convert a pyarrow Table batch into a list of PyG Data objects."""
        central_ids = batch_table.column('central_user_id').to_pylist()
        neighbor_ids = batch_table.column('neighbor_ids').to_pylist()
        retweeted_ids = batch_table.column('retweeted_ids').to_pylist()
        edge_srcs = batch_table.column('edge_src').to_pylist()
        edge_dsts = batch_table.column('edge_dst').to_pylist()
        labels = batch_table.column('label').to_pylist()

        data_list = []
        for i in range(len(central_ids)):
            all_ids = [central_ids[i]] + list(neighbor_ids[i])
            num_nodes = len(all_ids)
            user_ids = torch.tensor(all_ids, dtype=torch.long)
            retweeted_set = set(retweeted_ids[i])
            retweet_flag = torch.tensor(
                [1.0 if uid in retweeted_set else 0.0 for uid in all_ids],
                dtype=torch.float
            ).unsqueeze(1)
            src, dst = edge_srcs[i], edge_dsts[i]
            if src:
                edge_index = torch.tensor(
                    np.column_stack([src, dst]).astype(np.int64),
                    dtype=torch.long
                ).t().contiguous()
            else:
                edge_index = torch.zeros((2, 0), dtype=torch.long)
            label = torch.tensor(labels[i], dtype=torch.long)
            data_list.append(Data(
                user_ids=user_ids,
                retweet_flag=retweet_flag,
                edge_index=edge_index,
                y=label,
                num_nodes=num_nodes,
                central_mask=torch.zeros(num_nodes, dtype=torch.bool).index_fill_(0, torch.tensor([0]), True),
            ))
        return data_list

    def get_label_counts(self):
        """Scan all files to count labels (for class weighting)."""
        counts = {}
        for uid, path, n_rows in self.user_files:
            if self.fs is not None:
                with self.fs.open(path, 'rb') as f:
                    table = pq.read_table(f, columns=['label'])
            else:
                table = pq.read_table(path, columns=['label'])
            for label in table.column('label').to_pylist():
                counts[label] = counts.get(label, 0) + 1
            del table
        return counts


# ---------------------------------------------------------------------------
# Build train and val loaders from experiment manifest
# ---------------------------------------------------------------------------
# Train loader: train-group users' "train" split
_train_uids = [u["uid"] for u in experiment_manifest["train_users"]]
# Val loader: train-group users' "test" split + test-group users' both splits
_val_uids_test_split = [u["uid"] for u in experiment_manifest["train_users"]]
_val_uids_all = [u["uid"] for u in experiment_manifest["test_users"]]

train_loader = UserCacheGNNLoader(
    _train_uids, split="train", batch_size=BATCH_SIZE,
    shuffle=True, fs=_loader_fs,
)

# Validation: combine train-group test split + test-group both splits
# Use a composite loader for validation
class _CompositeUserCacheLoader:
    """Combines multiple UserCacheGNNLoaders into one iterable."""
    def __init__(self, loaders, max_samples=None):
        self.loaders = loaders
        self.max_samples = max_samples
        self._total_samples = sum(l.total_samples for l in loaders)
        if max_samples and self._total_samples > max_samples:
            self._effective_samples = max_samples
        else:
            self._effective_samples = self._total_samples

    @property
    def total_samples(self):
        return self._effective_samples

    @property
    def files(self):
        return [f for l in self.loaders for f in l.files]

    def __len__(self):
        return self._effective_samples // self.loaders[0].batch_size

    def __iter__(self):
        samples_yielded = 0
        for loader in self.loaders:
            for batch in loader:
                if self.max_samples and samples_yielded >= self.max_samples:
                    return
                yield batch
                samples_yielded += loader.batch_size

    def get_label_counts(self):
        counts = {}
        for loader in self.loaders:
            for k, v in loader.get_label_counts().items():
                counts[k] = counts.get(k, 0) + v
        return counts


_val_loader_train_test = UserCacheGNNLoader(
    _val_uids_test_split, split="test", batch_size=BATCH_SIZE,
    shuffle=False, fs=_loader_fs,
)
# Test-group users: both their train and test samples go to validation
_val_loader_test_train = UserCacheGNNLoader(
    _val_uids_all, split="train", batch_size=BATCH_SIZE,
    shuffle=False, fs=_loader_fs,
)
_val_loader_test_test = UserCacheGNNLoader(
    _val_uids_all, split="test", batch_size=BATCH_SIZE,
    shuffle=False, fs=_loader_fs,
)

val_loader = _CompositeUserCacheLoader(
    [_val_loader_train_test, _val_loader_test_train, _val_loader_test_test],
    max_samples=MAX_VAL_SAMPLES,
)

print(f"Train loader: {train_loader.total_samples} samples, "
      f"{len(train_loader)} batches, {len(train_loader.files)} user files")
print(f"Val loader:   {val_loader.total_samples} samples, "
      f"{len(val_loader)} batches, {len(val_loader.files)} user files")

# Compute class weights by scanning train labels
print("Computing class weights from train labels...")
label_counts = train_loader.get_label_counts()
total_train = sum(label_counts.values())
classes = np.array(sorted(label_counts.keys()))
class_weights = torch.tensor(
    [total_train / (len(classes) * label_counts[c]) for c in classes],
    dtype=torch.float
)
print(f"  Label counts: {label_counts}")
print(f"  Class weights: {class_weights.tolist()}")

# Free objects no longer needed
for _var in ['user_data', 'graph', 'processing_plan']:
    if _var in dir():
        exec(f'del {_var}')
gc.collect()
torch.cuda.empty_cache()

In [0]:
# Anti-overfitting: strong regularization to prevent memorizing train users' graph patterns.
# Test set has entirely unseen users — model must generalize graph structure, not memorize it.
model = RetweetGNN(
    ff_hidden_dim=64,
    gcn_hidden_dim=64,
    transformer_dim=64,
    transformer_heads=4,
    embeddings_path=EMBEDDINGS_PATH,
    device=device,
    dropout=0.5,           # high dropout forces redundant representations that transfer
    drop_edge_rate=0.1,    # drop edges — prevents memorizing specific neighbor patterns
).to(device)

# Only apply INIT_WEIGHTS_PATH when starting fresh (no existing checkpoint from this experiment).
# This prevents accidentally re-initializing from pre-trained weights when resuming a run.
_existing_checkpoint = os.path.join(EXPERIMENT_DIR, "best_retweet_gnn_general.pt")
_is_fresh_start = RESET_GNN_TRAINING or not os.path.exists(_existing_checkpoint)
_apply_init_weights = bool(INIT_WEIGHTS_PATH) and _is_fresh_start

if not _is_fresh_start and INIT_WEIGHTS_PATH:
    print(f"⚠️  Existing checkpoint found at {_existing_checkpoint} — skipping INIT_WEIGHTS_PATH.")
    print(f"   (Set RESET_GNN_TRAINING=True to force re-initialization from pre-trained weights.)")

if _apply_init_weights:
    print(f"Loading pre-trained weights from: {INIT_WEIGHTS_PATH}")
    state_dict = torch.load(INIT_WEIGHTS_PATH, map_location=device)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing:
        print(f"  Missing keys (will use random init): {missing}")
    if unexpected:
        print(f"  Unexpected keys (ignored): {unexpected}")
    print(f"  Weights loaded successfully. gate_param = {model.gate_param.item():.4f}")
else:
    # Gate at 0.0 → sigmoid=0.5 (balanced start). Let the model earn GNN contribution.
    # Shortcut head uses aggregate stats (rt_frac, node_count) which generalize better to unseen users.
    # If GNN can't beat shortcut on val, gate will stay low — that's fine.
    with torch.no_grad():
        model.gate_param.fill_(0.0)

In [0]:
gc.collect()
torch.cuda.empty_cache()

In [0]:
# Train with streaming DataLoaders — anti-overfitting configuration:
#   - lr=3e-3 with 5-epoch linear warmup: prevents fast memorization in early steps
#   - weight_decay=1e-3: L2 prevents weight specialization
#   - dropout=0.5 + drop_edge=0.1: stochastic regularization
#   - gate_param=0.0 (50/50): shortcut generalizes better; GNN must earn its contribution
#   - ParquetGNNLoader shuffles file order each epoch (no full in-memory shuffle needed)

import sys
from datetime import datetime
from contextlib import contextmanager

class TeeLogger:
    """Tee stdout to both the original stream and a timestamped log file."""
    def __init__(self, log_path, original_stdout):
        self._original = original_stdout
        self._file = open(log_path, "a", buffering=1)  # line-buffered
        self._line_buffer = ""

    def write(self, msg):
        self._original.write(msg)
        # Add timestamp at the start of each complete line
        for char in msg:
            if char == "\n":
                timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                self._file.write(f"[{timestamp}] {self._line_buffer}\n")
                self._line_buffer = ""
            else:
                self._line_buffer += char

    def flush(self):
        self._original.flush()
        self._file.flush()

    def close(self):
        # Flush any remaining buffer
        if self._line_buffer:
            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            self._file.write(f"[{timestamp}] {self._line_buffer}\n")
            self._line_buffer = ""
        self._file.close()

@contextmanager
def tee_to_log(log_path):
    """Context manager that tees all stdout to a timestamped log file."""
    original_stdout = sys.stdout
    tee = TeeLogger(log_path, original_stdout)
    sys.stdout = tee
    try:
        yield log_path
    finally:
        sys.stdout = original_stdout
        tee.close()
        print(f"Training log saved to: {log_path}")

TRAINING_LOG_PATH = f"{EXPERIMENT_DIR}/training.log"

# Reduce LR and weight decay when fine-tuning from pre-trained weights
# (only applies when init weights were actually loaded — i.e. fresh start)
_base_lr = 3e-3
_base_wd = 1e-3
_train_lr = _base_lr * FINETUNE_LR_FACTOR if _apply_init_weights else _base_lr
_train_wd = _base_wd * FINETUNE_WD_FACTOR if _apply_init_weights else _base_wd
_warmup_epochs = FINETUNE_WARMUP_EPOCHS if _apply_init_weights else 5
if _apply_init_weights:
    print(f"Fine-tuning mode:")
    print(f"  LR reduced from {_base_lr:.1e} to {_train_lr:.1e} (factor={FINETUNE_LR_FACTOR})")
    print(f"  Weight decay reduced from {_base_wd:.1e} to {_train_wd:.1e} (factor={FINETUNE_WD_FACTOR})")
    print(f"  Warmup epochs reduced from 5 to {_warmup_epochs}")

with tee_to_log(TRAINING_LOG_PATH):
    model, history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        experiment_dir=EXPERIMENT_DIR,
        class_weights=class_weights,
        epochs=EPOCHS,
        device=device,
        lr=_train_lr,
        log_every_n_steps=LOG_EVERY_N_STEPS,
        patience=PATIENCE,
        lr_warmup_epochs=_warmup_epochs,
        weight_decay=_train_wd,
        resume=False,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        mixed_precision=MIXED_PRECISION,
        train_f1_every_n_epochs=TRAIN_F1_EVERY_N_EPOCHS,
    )